# NB3 — Mispronunciation Detection (scoring + benchmark + product verdict)

Given a user recitation and the **expected verse**, this notebook:
1. phonemizes the expected diacritized verse -> canonical phoneme sequence (MSA phonetiser),
2. CTC-decodes the audio with the NB2 model -> predicted phoneme sequence,
3. aligns the two and **flags errors** (substitution / deletion / insertion), showing the correct phoneme,
4. maps each error back to the word it belongs to (for the app UI),
5. (optional) reports F1 / PER on the `QuranMB.v2` benchmark.

### Week-1 reality check (zero training)
Set `MODEL_DIR` to a **pre-trained Iqra'Eval model** (e.g. `IqraEval/Iqra_wav2vec2_base`)
to measure where an off-the-shelf phoneme model lands on YOUR audio before training
anything. Then point it at your NB2 `final/` to compare.

In [ ]:
!pip install -q "transformers==4.44.2" "datasets==2.21.0" soundfile librosa jiwer
# MSA phonetiser (inference-time G2P for arbitrary expected verses):
!git clone -q https://github.com/Iqra-Eval/MSA_phonetiser.git || echo 'already cloned'

In [ ]:
import os, sys, json, re
import numpy as np
import torch


MODEL_DIR = "/kaggle/input/datasets/qasimmaajid/model-3/w2vbert-quran-phoneme-real/final"   

SAMPLING_RATE = 16_000
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
PHONETISER_DIR = "MSA_phonetiser"
print("device:", DEVICE, "| model:", MODEL_DIR)
print("model dir exists:", os.path.isdir(MODEL_DIR))

In [ ]:
from transformers import AutoModelForCTC, AutoProcessor, Wav2Vec2BertForCTC, AutoFeatureExtractor, Wav2Vec2CTCTokenizer

# Load model + the pieces needed to decode CTC output to phoneme symbols.
model = AutoModelForCTC.from_pretrained(MODEL_DIR).to(DEVICE).eval()
try:
    processor = AutoProcessor.from_pretrained(MODEL_DIR)
    feature_extractor = processor.feature_extractor
    tokenizer = processor.tokenizer
except Exception:
    feature_extractor = AutoFeatureExtractor.from_pretrained(MODEL_DIR)
    tokenizer = Wav2Vec2CTCTokenizer.from_pretrained(MODEL_DIR)

id2tok = {i: t for t, i in tokenizer.get_vocab().items()}
blank_id = tokenizer.pad_token_id
print("vocab size:", len(id2tok), "| blank id:", blank_id)

In [ ]:
def recognise_phonemes(audio_array):
    """audio (float32 @16k) -> predicted phoneme list via CTC greedy decode.
    Silence tokens are dropped: <sil> is not a pronunciation error, and the
    model emits it between words, which would otherwise misalign vs the
    canonical sequence."""
    feats = feature_extractor(np.asarray(audio_array, dtype=np.float32),
                              sampling_rate=SAMPLING_RATE, return_tensors="pt")
    feats = {k: v.to(DEVICE) for k, v in feats.items()}
    with torch.no_grad():
        logits = model(**feats).logits[0]
    ids = logits.argmax(-1).tolist()
    out, prev = [], None
    for i in ids:
        if i != prev and i != blank_id:
            sym = id2tok.get(int(i), "<unk>")
            if sym not in ("<sil>", "sil"):
                out.append(sym)
        prev = i
    return out

In [ ]:
# ---- Canonical G2P: expected diacritized verse -> phoneme list ----------
sys.path.insert(0, PHONETISER_DIR)
_phonetise = None
try:
    from phonetiser import phonetise_Arabic
    _phonetise = phonetise_Arabic.phonetise
except Exception as e:
    print("phonetiser import failed, will fall back:", e)

def canonical_phonemes(tashkeel_text):
    """Return a flat phoneme list for a diacritized Arabic string.
    Uses the MSA phonetiser if available; output is space-separated symbols
    matching the training vocab convention (shadda=doubled symbol, long vowels aa/ii0/uu0).
    """
    if _phonetise is None:
        raise RuntimeError("MSA phonetiser unavailable; check the clone in cell 1.")
    _, with_spaces, _ = _phonetise(tashkeel_text)
    # with_spaces: list (per utterance) of space-separated phoneme strings
    line = with_spaces[0] if isinstance(with_spaces, (list, tuple)) else with_spaces
    return [t for t in str(line).split() if t]

# Smoke test
try:
    print(canonical_phonemes("بِسْمِ اللَّهِ الرَّحْمَٰنِ الرَّحِيمِ"))
except Exception as e:
    print("smoke test note:", e)

In [ ]:

model_vocab = set(id2tok.values()) - {"<pad>", "<unk>", "<s>", "</s>", "|", "<sil>"}

import re as _re
def _norm_one(p):
    if p in ("sil", "<sil>"):        # silence
        return None
    if p == "<":                      # elided word-initial hamza
        return None
    m = _re.fullmatch(r"([A-Za-z]+)0([A-Za-z]+)0", p)
    if m and m.group(1) == m.group(2):
        base = m.group(1)
        return base if len(base) > 1 else base + base   # i->ii, ii->ii
    m = _re.fullmatch(r"([A-Za-z]+)[0-9]", p)
    if m:
        return m.group(1)
    return p

def normalise_phonemes(seq):
    out = []
    for p in seq:
        q = _norm_one(p)
        if q:
            out.append(q)
    return out

probe = canonical_phonemes(
    "بِسْمِ اللَّهِ الرَّحْمَٰنِ الرَّحِيمِ الْحَمْدُ لِلَّهِ رَبِّ الْعَالَمِينَ"
)
probe_syms = set(normalise_phonemes(probe))
unknown = sorted(probe_syms - model_vocab)
print("model vocab size:", len(model_vocab))
print("canonical symbols after normalise:", sorted(probe_syms))
print("still-unknown symbols:", unknown)
print("OK\n" if not unknown else f">>> still unmapped: {unknown}\n")

In [ ]:
# ---- Alignment + error tagging (Levenshtein over phoneme symbols) -------
def align(ref, hyp):
    """Return list of ops: ('ok'|'sub'|'del'|'ins', ref_sym, hyp_sym).
    ref = canonical (expected), hyp = recognised (what was said).
    'del' = expected phoneme missing in speech (e.g. dropped shadda half).
    'sub' = wrong phoneme said (e.g. fatha instead of damma)."""
    n, m = len(ref), len(hyp)
    dp = [[0]*(m+1) for _ in range(n+1)]
    for i in range(n+1): dp[i][0] = i
    for j in range(m+1): dp[0][j] = j
    for i in range(1, n+1):
        for j in range(1, m+1):
            cost = 0 if ref[i-1] == hyp[j-1] else 1
            dp[i][j] = min(dp[i-1][j]+1, dp[i][j-1]+1, dp[i-1][j-1]+cost)
    i, j, ops = n, m, []
    while i > 0 or j > 0:
        if i > 0 and j > 0 and dp[i][j] == dp[i-1][j-1] + (0 if ref[i-1]==hyp[j-1] else 1):
            ops.append(("ok" if ref[i-1]==hyp[j-1] else "sub", ref[i-1], hyp[j-1])); i-=1; j-=1
        elif i > 0 and dp[i][j] == dp[i-1][j] + 1:
            ops.append(("del", ref[i-1], None)); i-=1
        else:
            ops.append(("ins", None, hyp[j-1])); j-=1
    return ops[::-1]

In [ ]:

MIN_ERRORS_PER_WORD = 1   # a word needs >=1 non-artifact error to be flagged

# consonants that can carry shadda from assimilation/gemination
_DOUBLED = {"ll","rr","nn","ss","SS","dd","DD","tt","TT","bb","mm","kk",
            "ff","qq","jj","zz","ZZ","hh","HH","xx","gg","**","cc","yy","ww"}

# vowels whose ONLY difference is length (short <-> long); same colour -> MINOR.
_VOWEL_LEN_PAIRS = {("a","aa"),("aa","a"),("i","ii"),("ii","i"),
                    ("u","uu"),("uu","u"),
                    ("A","AA"),("AA","A"),("I","II"),("II","I"),
                    ("U","UU"),("UU","U")}

def _collapse_gemination(seq):
    """Doubled consonant (ll, rr, SS ...) -> single, so sun-letter assimilation
    is never counted as a substitution. Long vowels (aa, ii, uu) untouched."""
    return [p[0] if p in _DOUBLED else p for p in seq]

def word_phonemes(w):
    """Canonical phonemes for one word, normalised to the model vocab."""
    return normalise_phonemes(canonical_phonemes(w))

def word_phonemes_connected(w):
    """Canonical phonemes with connected-speech elisions removed so it matches
    fluent mid-utterance recitation: drop leading hamzat-wasl '<' and 'al-' lam."""
    ph = list(word_phonemes(w))
    if ph and ph[0] == "<":                               # hamzat-wasl elides
        ph = ph[1:]
    if len(ph) >= 2 and ph[0] == "a" and ph[1] == "l":    # 'al-' article
        ph = ph[2:]
    return ph

def _severity(op, ref_sym, hyp_sym):
    """Classify one alignment op into 'minor' | 'major' | None (not an error)."""
    if op == "sub":
        if (ref_sym, hyp_sym) in _VOWEL_LEN_PAIRS:
            return "minor"          # vowel-length nuance (madd)
        return "major"              # wrong consonant or wrong vowel colour
    if op == "del":
        return "major"              # a phoneme was dropped
    return None

def analyse_recitation(audio_array, expected_tashkeel):
    """Per-word verdict with severity. Returns, per word: verdict
    ('pass'|'minor'|'major'), and the list of (severity, op, ref, hyp) errors.
    Pure insertions are ignored (matches the official scorer)."""
    words = expected_tashkeel.split()
    ref_flat, ref_word_of = [], []
    for wi, w in enumerate(words):
        ph = word_phonemes_connected(w)
        ref_flat += ph
        ref_word_of += [wi] * len(ph)
    hyp = recognise_phonemes(audio_array)

    # collapse gemination on BOTH sides so assimilation is never an error
    ops = align(_collapse_gemination(ref_flat), _collapse_gemination(hyp))

    errors = {wi: [] for wi in range(len(words))}
    rp = 0
    for op, r, h in ops:
        if op in ("ok", "sub", "del"):
            wi = ref_word_of[rp] if rp < len(ref_word_of) else len(words) - 1
            orig_ref = ref_flat[rp] if rp < len(ref_flat) else r
            sev = _severity(op, orig_ref, h)
            if sev:
                errors[wi].append((sev, op, orig_ref, h))
            rp += 1
        # pure insertions (op == 'ins') are NOT counted

    report = []
    for wi, w in enumerate(words):
        errs = errors[wi]
        if any(s == "major" for s, *_ in errs):
            verdict = "major"
        elif errs:
            verdict = "minor"
        else:
            verdict = "pass"
        report.append({
            "word": w,
            "verdict": verdict,
            "errors": errs,
            "correct_phonemes": word_phonemes_connected(w),
        })
    return report

In [ ]:
# ---- Demo on one clip ---------------------------------------------------
import soundfile as sf, subprocess
import jiwer

def load_audio(path):
    if path.endswith((".opus", ".mp3", ".m4a")):
        cmd = ["ffmpeg","-loglevel","error","-i",path,"-f","f32le","-ar",str(SAMPLING_RATE),"-ac","1","pipe:1"]
        raw = subprocess.run(cmd, capture_output=True).stdout
        return np.frombuffer(raw, dtype=np.float32).copy()
    a, sr = sf.read(path, dtype="float32")
    return a

DEMO_AUDIO = "/kaggle/input/datasets/qasimmaajid/fatiha-clip/audio.opus"   # EDIT
DEMO_VERSE = "بِسْمِ اللَّهِ الرَّحْمَٰنِ الرَّحِيمِ الْحَمْدُ لِلَّهِ رَبِّ الْعَالَمِينَ الرَّحْمَٰنِ الرَّحِيمِ مَالِكِ يَوْمِ الدِّينِ إِيَّاكَ نَعْبُدُ وَإِيَّاكَ نَسْتَعِينُ اهْدِنَا الصِّرَاطَ الْمُسْتَقِيمَ صِرَاطَ الَّذِينَ أَنْعَمْتَ عَلَيْهِمْ غَيْرِ الْمَغْضُوبِ عَلَيْهِمْ وَلَا الضَّالِّينَ"

if os.path.exists(DEMO_AUDIO):
    audio = load_audio(DEMO_AUDIO)
    print(f"audio: {len(audio)/SAMPLING_RATE:.1f}s\n")

    heard    = recognise_phonemes(audio)
    expected = normalise_phonemes(canonical_phonemes(DEMO_VERSE))
    print("EXPECTED phonemes :", " ".join(expected))
    print("MODEL HEARD       :", " ".join(heard))


    per = jiwer.wer(" ".join(expected), " ".join(heard))
    print(f"\n>>> Phoneme accuracy on this clip: {1-per:.1%}  (PER {per:.3f})")
    print(f"    Old whisper-medium on the same clip: ~31% correct-word-won.\n")


    rep = analyse_recitation(audio, DEMO_VERSE)
    TAG = {"pass": "ok   ", "minor": "MINOR", "major": "MAJOR"}
    n_pass  = sum(r["verdict"] == "pass"  for r in rep)
    n_minor = sum(r["verdict"] == "minor" for r in rep)
    n_major = sum(r["verdict"] == "major" for r in rep)
    for r in rep:
        errs = "; ".join(f"{sev}:{op}:{rr}->{hh}" for sev,op,rr,hh in r["errors"]) or "-"
        print(f"[{TAG[r['verdict']]}] {r['word']:15s} correct={' '.join(r['correct_phonemes'])}  {errs}")

    clean = n_pass + n_minor   # words with no MAJOR (letter) error
    print(f"\nper-word: {n_pass} correct, {n_minor} minor (vowel-length), "
          f"{n_major} major (letter) errors  —  {clean}/{len(rep)} free of letter errors "
          f"({clean/len(rep):.0%}).")
    print("MAJOR = wrong/dropped letter (alarm).  MINOR = madd/length nuance (soft note).")
else:
    print("Set DEMO_AUDIO to a real clip to run the demo.")

In [ ]:

from datasets import load_dataset
import jiwer

RUN_BENCHMARK = False   # set True once you have the test set + (optional) GT
if RUN_BENCHMARK:
    qmb = load_dataset("IqraEval/QuranMB.v2", split="test")
    print(qmb)
    pers = []
    for ex in qmb:
        hyp = recognise_phonemes(ex["audio"]["array"])
        # If a canonical/tashkeel field exists, phonemize it; else skip PER.
        ref_text = ex.get("tashkeel_sentence") or ex.get("sentence")
        if ref_text:
            ref = canonical_phonemes(ref_text)
            pers.append(jiwer.wer(" ".join(ref), " ".join(hyp)))
    if pers:
        print(f"PER vs canonical: {np.mean(pers):.4f} over {len(pers)} clips")
    print("For detection F1 (TA/FR/FA/CD), align hyp against BOTH the canonical and "
          "the annotated-verbatim phoneme sequences from the GT set.")

### Notes
- **Precision tuning** (`MIN_ERRORS_PER_WORD`) is your main product dial: raise it so the
  app stays quiet unless confident — important for a Quran app where false flags are costly.
- **Detection F1 on QuranMB** needs the gated GT phoneme annotations; accept the dataset
  terms on HuggingFace and re-run with `HF_TOKEN` set.
- The phonetiser output convention must match the training vocab; if symbols differ
  (e.g. subscripts), add a small mapping table here mirroring AraS2P Table 2.